# minutes term — P(>=60' | played)

**Render-not-decide.** A re-runnable view of the `minutes` hurdle, **not** the record — the frozen numbers live in [predictive-phase3-points-model.md](../../../docs/studies/results/predictive-phase3-points-model.md). Logic is in `model/terms/minutes/` (+ the shared binary base).

A **logistic hurdle** on `play60 = 1{minutes >= 60}`: outfield per-position; **GK overridden** with a robust rate (near-constant target). `p60` feeds appearance points (`1 + p60`) and gates clean_sheet. Ranking is ~parity outfield by design (the value is a calibrated probability). See `ASSUMPTIONS.md`.

## Setup

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

from dal.pipeline import load as load_mart
from model.terms.minutes import MinutesHurdleModel, MinutesTerm

mart = load_mart()
term = MinutesTerm(MinutesHurdleModel(variant="selected"))
term.model.pool.minimal, [f.name for f in term.model.pool.candidates]

## Pre-fit assumptions
> Binary term: family is logistic by construction; the floor is class balance (>=60' base rate) + enough events. See `ASSUMPTIONS.md` §1.

In [ ]:
report = term.model.check_assumptions(MinutesHurdleModel.population(mart))
print(f"detectable={report.detectable}  n_train={report.n_train}")
report.dispersion  # base_rate / n_events / family

## Gate — p60 vs the lagged minutes level (minutes_roll3)
> Outfield ranking is ~parity by design (calibration, not a ranking win); GK is near-constant (~0.99 base rate). Read the value as a calibrated probability.

In [ ]:
gate = term.validate(mart)
display(gate.table)

ax = gate.table.set_index("position")[["baseline", "p60"]].plot.bar(figsize=(7, 3.5))
ax.set_ylabel("within-position Spearman"); ax.set_title("minutes hurdle: model vs lagged-minutes baseline"); plt.tight_layout()

## Calibration view — is p60 a good probability?
> The point of the hurdle is a calibrated P(>=60'), not a ranking. Bin p60 and compare to the realized >=60' rate.

In [ ]:
fitted = term.model.fit(mart)
pop = MinutesHurdleModel.population(mart).assign(p60=fitted.predictions)
ev = pop[pop['p60'].notna()].copy()
ev['bucket'] = (ev['p60'] * 10).round() / 10
cal = ev.groupby('bucket').agg(predicted=('p60', 'mean'), realized=('play60', 'mean'), n=('play60', 'size'))
display(cal)
ax = cal.plot(y=['predicted', 'realized'], marker='o', figsize=(5, 4), title='minutes hurdle calibration')
ax.set_xlabel('p60 bucket'); plt.tight_layout()